# part5.ipynb — Guardrail pipeline

**Prerequisite:** Run `part1`–`part4` (for mitigated weights + `isotonic_calibrator.joblib`). `pipeline.py` must be in this directory.


### i220524 FAST-NUCES Assignment 2 — Responsible & Explainable AI


## Environment setup

Install dependencies (local or Colab), then continue. On Colab: **Runtime → Change runtime type → GPU**.



In [ ]:
"""Install dependencies via `pip install -r requirements.txt` in a fresh environment (e.g. Colab)."""


def ensure_pkg():
    """No-op placeholder; run pip manually or automate here if desired."""
    return None


ensure_pkg()


In [ ]:
"""Imports and global configuration for reproducibility.

Sets USE_TF=0 before HuggingFace loads, and forces trainer_utils.is_tf_available() to False so Trainer.setup never imports TensorFlow (broken TF wheels still report as installed).
"""
import os
os.environ["USE_TF"] = "0"
import re
import random
import zipfile
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import transformers.trainer_utils as _hf_trainer_utils
_hf_trainer_utils.is_tf_available = lambda: False
from datasets import Dataset

from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ZIP_PATH = os.path.join(os.getcwd(), "jigsaw-unintended-bias-in-toxicity-classification.zip")
TRAIN_CSV_NAME = "train.csv"
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

BASELINE_DIR = os.path.join(ARTIFACT_DIR, "baseline_distilbert")
POISON_DIR = os.path.join(ARTIFACT_DIR, "poisoned_distilbert")
REWEIGHT_DIR = os.path.join(ARTIFACT_DIR, "reweighted_distilbert")
OVERSAMPLE_DIR = os.path.join(ARTIFACT_DIR, "oversample_distilbert")
ISOTONIC_PATH = os.path.join(ARTIFACT_DIR, "isotonic_calibrator.joblib")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

"""Optional: set True only for quick smoke tests (not for submission)."""
FAST_DEBUG = False
if FAST_DEBUG:
    NUM_EPOCHS = 1


## Load data from the zip (stratified 100k train / 20k eval)

We read only the columns needed to limit memory. **Label:** `y = (target >= 0.5).astype(int)`. `train_test_split` returns `(larger_slice, test_size_slice)`; the 20k holdout is the **second** return value.



In [ ]:
"""Load `train.csv` from the competition zip without extracting the full archive to disk."""


def read_train_from_zip(zip_path: str, csv_name: str) -> pd.DataFrame:
    """Read selected columns for memory efficiency and normalize names."""
    usecols = [
        "comment_text",
        "target",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian",
    ]
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, usecols=usecols)
    df = df.rename(columns={"target": "toxic"})
    df["comment_text"] = df["comment_text"].fillna("").astype(str)
    for c in ["toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df["y"] = (df["toxic"] >= 0.5).astype(int)
    return df


df_all = read_train_from_zip(ZIP_PATH, TRAIN_CSV_NAME)
if FAST_DEBUG:
    eval_n = 2000
    train_n = 5000
    cap = eval_n + train_n + 5000
    df_all, _ = train_test_split(
        df_all, train_size=min(cap, len(df_all)), stratify=df_all["y"], random_state=SEED
    )
else:
    eval_n = 20000
    train_n = 100000

y = df_all["y"].values
train_remaining, eval_df = train_test_split(
    df_all, test_size=eval_n, stratify=y, random_state=SEED
)
y_rem = train_remaining["y"].values
train_df, _discard = train_test_split(
    train_remaining, train_size=train_n, stratify=y_rem, random_state=SEED
)

print("train_df:", train_df.shape, "eval_df:", eval_df.shape)
print("toxic rate train:", train_df["y"].mean(), "eval:", eval_df["y"].mean())


In [ ]:
"""Device and mitigated checkpoint for Part 5 (run part1–part4 for full artifacts)."""
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


BEST_MITIGATED_DIR = BASELINE_DIR
for _d in (REWEIGHT_DIR, OVERSAMPLE_DIR, BASELINE_DIR):
    if os.path.isfile(os.path.join(_d, "config.json")):
        BEST_MITIGATED_DIR = _d
        break


def hf_predict_proba_toxic(texts, m, tok, device_obj, max_len):
    """Return toxic-class probabilities for a list of strings."""
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device_obj)
    with torch.no_grad():
        logits = m(**enc).logits
        pr = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    return pr


# Part 5 — Guardrail pipeline (`pipeline.py`)

`ModerationPipeline.predict` applies the regex blocklist, then **isotonic-calibrated** transformer probabilities, then routes **0.4–0.6** to human review.

The notebook fits `CalibratedClassifierCV(..., method='isotonic')` for reporting, then saves a standalone `IsotonicRegression` mapping for `pipeline.py` inference (avoids pickling duplicate transformer weights inside sklearn).



In [ ]:
"""Demonstrate `ModerationPipeline` on 1000 evaluation comments + layer distribution plots."""
from pipeline import ModerationPipeline, input_filter, HFProbabilityEstimator

mitigated_tokenizer = AutoTokenizer.from_pretrained(BEST_MITIGATED_DIR)
mitigated_model = AutoModelForSequenceClassification.from_pretrained(BEST_MITIGATED_DIR)
mitigated_model.eval()
mitigated_model.to(device)

calib_eval_idx = eval_df.sample(n=min(3000, len(eval_df)), random_state=SEED).index
calib_texts = eval_df.loc[calib_eval_idx, "comment_text"].tolist()
calib_y = eval_df.loc[calib_eval_idx, "y"].to_numpy()
raw_cal_scores = hf_predict_proba_toxic(calib_texts, mitigated_model, mitigated_tokenizer, device, MAX_LENGTH)

base_est = HFProbabilityEstimator(mitigated_model, mitigated_tokenizer, device, MAX_LENGTH)
cc = CalibratedClassifierCV(base_est, method="isotonic", cv="prefit")
cc.fit(calib_texts, calib_y)

iso = IsotonicRegression(out_of_bounds="clip")
iso.fit(raw_cal_scores, calib_y)
joblib.dump(iso, ISOTONIC_PATH)

PIPE_MODEL_DIR = os.path.join(ARTIFACT_DIR, "best_mitigated_for_pipeline")
mitigated_model.save_pretrained(PIPE_MODEL_DIR)
mitigated_tokenizer.save_pretrained(PIPE_MODEL_DIR)

pipe = ModerationPipeline(model_dir=PIPE_MODEL_DIR, calibrator_path=ISOTONIC_PATH, device=str(device))

demo_n = min(1000, len(eval_df))
demo_df = eval_df.sample(n=demo_n, random_state=SEED)

counts = {"input_filter": 0, "model_block": 0, "model_allow": 0, "model_review": 0}
cat_counts = {
    "direct_threat": 0,
    "self_harm_directed": 0,
    "doxxing_stalking": 0,
    "dehumanization": 0,
    "coordinated_harassment": 0,
}

for t in demo_df["comment_text"].tolist():
    r = input_filter(t)
    if r is not None:
        counts["input_filter"] += 1
        cat = r.get("category")
        if cat in cat_counts:
            cat_counts[cat] += 1
        continue
    d = pipe.predict(t)
    if d["decision"] == "block":
        counts["model_block"] += 1
    elif d["decision"] == "allow":
        counts["model_allow"] += 1
    else:
        counts["model_review"] += 1

print("Layer counts:", counts)
print("Regex category counts:", cat_counts)

plt.figure(figsize=(6, 6))
plt.pie(list(counts.values()), labels=list(counts.keys()), autopct="%1.1f%%")
plt.title("Fraction of decisions by layer (demo)")
plt.show()

auto_texts = []
auto_true = []
review_true = []
for txt, yt in zip(demo_df["comment_text"].tolist(), demo_df["y"].tolist()):
    if input_filter(txt) is not None:
        continue
    d = pipe.predict(txt)
    if d["decision"] in ("block", "allow"):
        auto_texts.append(txt)
        auto_true.append(yt)
    else:
        review_true.append(yt)

if auto_texts:
    y_auto_true = np.array(auto_true)
    y_auto_hat = []
    p_obj = ModerationPipeline(
        model_dir=PIPE_MODEL_DIR, calibrator_path=ISOTONIC_PATH, device=str(device)
    )
    for txt in auto_texts:
        pr = p_obj._calibrated_toxic_prob(txt)
        y_auto_hat.append(1 if pr >= 0.6 else 0)
    y_auto_hat = np.array(y_auto_hat)
    print(
        "Auto-actioned F1/precision/recall:",
        f1_score(y_auto_true, y_auto_hat, pos_label=1),
        precision_score(y_auto_true, y_auto_hat, pos_label=1, zero_division=0),
        recall_score(y_auto_true, y_auto_hat, pos_label=1, zero_division=0),
    )

if review_true:
    print("Review queue toxic rate (ground truth):", float(np.mean(review_true)))


In [ ]:
"""Uncertainty band sensitivity analysis (required)."""


def pipeline_counts(block_hi: float, allow_lo: float):
    """Count review vs auto model decisions excluding regex blocks."""
    p = ModerationPipeline(
        model_dir=PIPE_MODEL_DIR,
        calibrator_path=ISOTONIC_PATH,
        device=str(device),
        block_hi=block_hi,
        allow_lo=allow_lo,
    )
    rev = 0
    auto = 0
    for txt in demo_df["comment_text"].tolist():
        if input_filter(txt) is not None:
            continue
        d = p.predict(txt)
        if d["decision"] == "review":
            rev += 1
        elif d["decision"] in ("block", "allow"):
            auto += 1
    return rev, auto


def auto_f1_at(block_hi: float, allow_lo: float) -> float:
    """F1 on toxic class for auto block/allow using block threshold on calibrated scores."""
    p = ModerationPipeline(
        model_dir=PIPE_MODEL_DIR,
        calibrator_path=ISOTONIC_PATH,
        device=str(device),
        block_hi=block_hi,
        allow_lo=allow_lo,
    )
    ys = []
    yh = []
    for txt, yt in zip(demo_df["comment_text"].tolist(), demo_df["y"].tolist()):
        if input_filter(txt) is not None:
            continue
        d = p.predict(txt)
        if d["decision"] not in ("block", "allow"):
            continue
        pr = p._calibrated_toxic_prob(txt)
        ys.append(yt)
        yh.append(1 if pr >= block_hi else 0)
    if not ys:
        return float("nan")
    return f1_score(np.array(ys), np.array(yh), pos_label=1)


configs = [
    ("default 0.4–0.6", 0.6, 0.4),
    ("narrow 0.45–0.55", 0.55, 0.45),
    ("wide 0.3–0.7", 0.7, 0.3),
]

rows_sens = []
for name, hi, lo in configs:
    rev, auto = pipeline_counts(hi, lo)
    rows_sens.append(
        {
            "band": name,
            "review_count": rev,
            "auto_count": auto,
            "auto_f1_toxic": auto_f1_at(hi, lo),
        }
    )

print(pd.DataFrame(rows_sens))


### Part 5 — Uncertainty band choice (model answer — cite your sensitivity table)

The **0.4–0.6** band routes **only middling calibrated probabilities** to humans; **≤0.4** allow and **≥0.6** block stay automatic. That is a **middle ground**: wide enough to catch model doubt after **isotonic calibration**, not so wide that almost everything hits review.

**Narrow (e.g. 0.45–0.55):** fewer **review_count** rows, but more borderline cases are **auto-blocked or auto-allowed**—risky if calibration is imperfect. **Wide (e.g. 0.3–0.7):** **review_count** grows (cost, latency), but fewer decisions are forced when the model is unsure; **auto_f1_toxic** may shift because the auto subset changes.

**Default choice:** Keep **0.4–0.6** unless your table shows **auto_f1** collapsing at default while narrow improves it **and** ops can absorb mis-automation risk—or unless review volume is untenable and you **narrow** with extra safeguards (sampling, appeals). **Edit:** One sentence comparing **review_count** and **auto_f1_toxic** across the three rows you printed.

